Project 1 — Banking domain (your "POC-level" project)
Intelligent Banking Assistant with RAG + Agentic Routing
Build a multi-agent system where a user can ask banking queries — loan eligibility, fraud alerts, transaction summaries — and the system routes them intelligently between a RAG agent (policy documents), a SQL agent (transaction data), and a summarization agent. Deploy end-to-end with Docker + Streamlit on HuggingFace Spaces with CI/CD via GitHub Actions.

Why it stands out: combines RAG + agentic AI + your domain expertise + full deployment. It directly tells the story of "I know banking AND I know GenAI AND I can ship." New tool to explore: LangGraph for multi-agent orchestration — this is becoming the industry standard and will impress interviewers.

### Import necessary libraries

In [ ]:
!pip install langchain langchain-community langchain-groq langgraph \
             faiss-cpu sentence-transformers python-dotenv \
             sqlalchemy pandas groq


### Connect to LLM

In [ ]:
import os
from langchain_groq import ChatGroq

# (In production/VS Code, we load from .env — never hardcode in real repos)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # Load from .env

# Initialise the LLM
llm = ChatGroq(
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # Load from .env
    model_name="llama-3.1-8b-instant",
    temperature=0
)

#Why `temperature=0`?**
#For a banking assistant we want consistent, factual answers — not creative ones. Temperature 0 means the model always picks the most confident response. Worth mentioning in interviews!

# Quick test
response = llm.invoke("Say 'LLM connected successfully' and nothing else.")
print(response.content)

LLM connected successfully.


**Observation** :

I chose Llama 3.1 8B Instant on Groq because the task is retrieval-augmented — the model doesn't need to reason deeply, just understand queries and summarise retrieved context. The instant variant gives low latency which matters for a real-time banking assistant, and Groq's LPU hardware makes it significantly faster than GPU-based inference.

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 8.6 MB/s eta 0:00:00


### Load PDF documents

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Your PDF path will be:
DOCS_PATH = "/content/drive/MyDrive/AIML GL/Projects/Banking Intelligence Assistance/Documents/"

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Path to your PDFs in Google Drive
DOCS_PATH = "/content/drive/MyDrive/AIML GL/Projects/Banking Intelligence Assistance/Documents/"

# All 6 PDFs
PDF_FILES = [
    "hdfc_credit_card_policy.pdf",
    "hdfc_customer_compensation_policy.pdf",
    "hdfc_grievance_policy.pdf",
    "hdfc_personal_loan_agreement.pdf",
    "hdfc_savings_account_charges.pdf",
    "hdfc_general_terms_conditions.pdf"
]

# Load all PDFs
all_documents = []
for pdf in PDF_FILES:
    full_path = os.path.join(DOCS_PATH, pdf);
    try:
        loader = PyPDFLoader(full_path)
        docs = loader.load()
        all_documents.extend(docs)
        print(f" Loaded: {pdf} — {len(docs)} pages")
    except Exception as e:
        print(f" Failed to load {pdf}: {e}")

print(f"\n Total pages loaded: {len(all_documents)}")

 Loaded: hdfc_credit_card_policy.pdf — 6 pages
 Loaded: hdfc_customer_compensation_policy.pdf — 5 pages
 Loaded: hdfc_grievance_policy.pdf — 16 pages
 Loaded: hdfc_personal_loan_agreement.pdf — 16 pages
 Loaded: hdfc_savings_account_charges.pdf — 2 pages
 Loaded: hdfc_general_terms_conditions.pdf — 6 pages

 Total pages loaded: 51


### Splitting the PDF data

In [ ]:
# Split documents into chunks
import re

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(all_documents)
print(f" Chunks before cleaning: {len(chunks)}")

# Clean chunks
def clean_text(text):
    text = re.sub(r'Classification\s*[-–]\s*Internal', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\s{3,}', ' ', text)
    text = re.sub(r'as on \d{2}\.\d{2}\.\d{4}', '', text)
    return text.strip()

for chunk in chunks:
    chunk.page_content = clean_text(chunk.page_content)

# Remove noise chunks
chunks = [chunk for chunk in chunks if len(chunk.page_content) > 50]
print(f" Chunks after cleaning: {len(chunks)}")
print(f"\n--- Sample chunk ---")
print(chunks[10].page_content)

 Chunks before cleaning: 578
 Chunks after cleaning: 577

--- Sample chunk ---
Repayment Through any of the payment channels (E.g. Net banking, ATMs, 
Cheque /cash deposit in branches, Standing instructions for 
account holders with HDFC Bank). For credit cards issued to NRIs / 
PIOs, the repayment shall be as per the defi ned process for NRI 
sourcing and FEMA directions issued by RBI from time to time. Security / Collateral  No collateral required for regular Cards. There is a FD based card 
where the Credit card is issued with a lien marked against the Fixed


**Observation :**

Why these specific settings? — Great interview talking point:

- chunk_size=500 — Each chunk is ~500 characters. Small enough to be precise when retrieving, large enough to contain meaningful context. Too large and retrieval becomes noisy; too small and you lose context.

 - chunk_overlap=50 — 50 characters of overlap between consecutive chunks. This ensures that if an important sentence falls at the boundary of two chunks, it still gets captured in at least one of them.

 - separators — We split preferentially on paragraph breaks first (\n\n), then line breaks, then sentences. This keeps semantically related text together rather than cutting mid-sentence.

We use chunk-level **embeddings**, not word-level. Each chunk is converted into a **384**-**dimensional** dense vector that captures its semantic meaning. This allows FAISS to retrieve **contextually** **relevant** chunks even when the user's query uses completely different words than what's in the document — because similar meanings produce geometrically close vectors in the embedding space."

### Building the FAISS vector store

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print(" Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(" Rebuilding FAISS index with all 6 PDFs...")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f" FAISS index rebuilt!")
print(f" Total vectors indexed: {vectorstore.index.ntotal}")

 Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Rebuilding FAISS index with all 6 PDFs...
 FAISS index rebuilt!
 Total vectors indexed: 577


**Observation:**

Why **all-MiniLM-L6-v2**? — Another great interview point:

 - It's a lightweight but highly capable sentence embedding model — only 80MB
 - Specifically trained for semantic similarity, meaning it understands that "credit card payment" and "outstanding balance due" are related concepts
 - Runs entirely locally — no API calls, no cost, no latency for embeddings
 - Industry standard choice for RAG prototypes

What's happening under the hood:

Each of the 164 chunks gets converted into a 384-dimensional vector (a list of numbers representing its meaning). FAISS stores all these vectors in a way that allows blazing-fast similarity search — when a user asks a question, their question also becomes a vector and FAISS finds the closest matching chunks in milliseconds.

### Testing the retrieval

In [ ]:
# Test retrieval — does FAISS find the right chunks?
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 most relevant chunks
)

# Test query
test_query = "What is the late payment fee for credit cards?"
results = retriever.invoke(test_query)

print(f" Query: {test_query}")
print(f" Top {len(results)} chunks retrieved:\n")
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

 Query: What is the late payment fee for credit cards?
 Top 3 chunks retrieved:

--- Chunk 1 ---
reference to a merchant establishment will be handled as per chargeback rules laid down by VISA / Mastercard 
/ Diners / Rupay network. The Bank will provide explanation and, if necessary, documentary evidence to the 
customer within a maximum period of sixty days. Failure on the part of the card-issuers to complete the process of closure within seven working days shall result 
in a penalty of ₹500 per day of delay payable to the customer, till the closure of the credit card provided there

--- Chunk 2 ---
transparency and customer convenience • The credit cards are issued by the Bank at the sole discretion of the Bank and subject to 
adherence of Credit norms and documentation requirements of the Bank. • The credit limit for a cardholder shall be assessed by the Bank in accordance with the 
underwriting and Credit guidelines formulated by the Bank from time to time for underwriting 
of the

In [ ]:
# Let's search directly in the raw text to see if the info exists
print(" Searching raw chunks for 'late payment'...\n")

matches = [chunk for chunk in chunks if "late payment" in chunk.page_content.lower()]

print(f"Found {len(matches)} chunks mentioning 'late payment':\n")
for i, match in enumerate(matches):
    print(f"--- Match {i+1} ---")
    print(match.page_content)
    print()

 Searching raw chunks for 'late payment'...

Found 1 chunks mentioning 'late payment':

--- Match 1 ---
above and for reasons directly attributable to the Bank, the customer shall be compensated at the prevailing 
Savings interest rate for the period between the due date of direct / EC S debit and the date of actual debit 
carried out by the bank. Direct / ECS debits which are towards payments of an Equated Monthly Instalment 
(EMI), the Bank would reimburse the customer, penal interest, late payment charges, if any levied upon



**Observation -**

Now we know exactly what's happening — the information exists but is **incomplete**. The chunk mentions "late payment charges" but doesn't state the specific fee amount. This is a data coverage issue, not a retrieval bug. The retriever is actually working correctly!
This is a realistic RAG scenario — in production, documents are always imperfect.

Here's how we handle it professionally:
Fix 1 — Switch to MMR search (better retrieval diversity).

### Fix 1 — Switch to MMR search

**Observation -**

Why **MMR** over simple similarity search? — Strong interview answer:

Simple similarity search often returns 3 nearly identical chunks — lots of redundancy, little new information. MMR solves this by penalising chunks that are too similar to ones already selected, giving you broader, more useful context for the LLM to work with.

### Building the RAG agent chain

In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7}
)

# Grounded prompt
prompt_template = """You are a helpful HDFC Bank policy assistant.
Use ONLY the context below to answer the customer's question.
If the answer is not in the context, say "I don't have enough information
in the policy documents to answer this. Please contact HDFC Bank directly."

Context:
{context}

Customer Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Helper to format retrieved chunks into single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(" RAG Agent ready with LCEL!")

 RAG Agent ready with LCEL!


**Observation -**

**Why LCEL is better** — good interview point:
LCEL uses a pipe | operator to chain components, making the data flow explicit and readable. It's also more flexible — easier to swap components, add middleware, and integrate with LangSmith for tracing. RetrievalQA is now considered legacy.

#### Testing the queries

In [30]:
# Test queries covering all 6 PDFs
test_queries = [
    "What is the minimum balance required for a savings account?",
    "What happens if I miss my loan EMI payment?",
    "How can I raise a grievance against HDFC Bank?",
    "What is the interest rate on revolving credit for classic cards?",
    "What documents are needed for KYC verification?"
]

print(" Testing RAG Agent across all policy areas:\n")
for query in test_queries:
    result = rag_chain.invoke(query)  # LCEL takes plain string, not dict
    print(f" {query}")
    print(f" {result}")  # LCEL returns string directly, not dict
    print()

 Testing RAG Agent across all policy areas:

 What is the minimum balance required for a savings account?
 The minimum balance requirement for a savings account varies based on the branch location. 

For Metro and Urban branches, the minimum balance requirement is either `10,000/- or an FD of `1 Lac for a minimum 1 year 1 day period.

For Semi Urban branches, the minimum balance requirement is either `5,000/- or an FD of `50,000 for a minimum 1 year 1 day period.

For Rural branches, the minimum balance requirement is either `2,500/- or an FD of `25,000/- for a minimum 1 year 1 day period.

 What happens if I miss my loan EMI payment?
 If you miss your loan EMI payment, you will be charged an Overdue EMI interest of 2% per month on the EMI amount.

 How can I raise a grievance against HDFC Bank?
 You can raise a grievance against HDFC Bank by contacting the Grievance Redressal Officer, Mrs. Deepa Balakrishnan, at +91 22 61606160. Alternatively, you can also contact the Call Centres at 

**Observations -**

The response received from the RAG is good and relevant:

-  Savings account — correctly differentiates Metro/Urban/Semi-Urban/Rural minimums
-  Loan EMI — specific 2% per month overdue interest
-  Grievance — real officer name, phone number, and Chennai address
-  Credit card interest — exact 3.75%/month (45% per annum)
-  KYC documents — accurate list from the policy

## Building the SQL Agent

In [31]:
import sqlite3
import random
from datetime import datetime, timedelta

# Create SQLite database
conn = sqlite3.connect('banking.db')
cursor = conn.cursor()

# Drop existing tables to start fresh
cursor.execute("DROP TABLE IF EXISTS transactions")
cursor.execute("DROP TABLE IF EXISTS credit_cards")
cursor.execute("DROP TABLE IF EXISTS customers")

# ── TABLE 1: CUSTOMERS ──
cursor.execute('''
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    name TEXT,
    account_type TEXT,
    balance REAL,
    credit_score INTEGER,
    branch TEXT,
    email TEXT,
    phone TEXT
)
''')

# ── TABLE 2: TRANSACTIONS ──
cursor.execute('''
CREATE TABLE transactions (
    txn_id TEXT PRIMARY KEY,
    customer_id TEXT,
    date TEXT,
    amount REAL,
    type TEXT,
    merchant TEXT,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
''')

# ── TABLE 3: CREDIT CARDS ──
cursor.execute('''
CREATE TABLE credit_cards (
    card_id TEXT PRIMARY KEY,
    customer_id TEXT,
    card_type TEXT,
    credit_limit REAL,
    outstanding REAL,
    minimum_due REAL,
    due_date TEXT,
    status TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
''')

print(" Tables created!")

# ── SEED DATA ──
random.seed(42)

first_names = ["Sushmita", "Rahul", "Priya", "Arjun", "Deepa", "Vikram",
               "Ananya", "Rohan", "Kavya", "Suresh", "Meera", "Kiran",
               "Sanjay", "Divya", "Arun", "Nisha", "Rajesh", "Pooja",
               "Harish", "Lakshmi", "Amit", "Sunita", "Ganesh", "Rekha",
               "Vijay", "Swati", "Manoj", "Asha", "Ravi", "Geeta",
               "Naveen", "Usha", "Prakash", "Shobha", "Dinesh", "Radha",
               "Sunil", "Vasantha", "Mohan", "Saritha", "Ajay", "Padma",
               "Venkat", "Hema", "Rajan", "Geetha", "Kumar", "Vani",
               "Balaji", "Nirmala"]

last_names = ["Sharma", "Mehta", "Nair", "Patel", "Krishnan", "Reddy",
              "Iyer", "Pillai", "Joshi", "Gupta", "Singh", "Kumar",
              "Rao", "Shah", "Verma", "Menon", "Chatterjee", "Mishra",
              "Bhat", "Kaur", "Das", "Srinivasan", "Hegde", "Desai",
              "Naidu", "Rajan", "Subramanian", "Murthy", "Pandey", "Nambiar"]

branches = ["Metro", "Metro", "Metro", "Urban", "Urban", "Semi Urban", "Rural"]
account_types = ["Savings", "Savings", "Savings", "Current"]
card_types = ["Classic", "Classic", "Platinum", "Titanium"]
card_statuses = ["Active", "Active", "Active", "Active", "Overdue", "Blocked"]

merchants = ["Swiggy", "Amazon", "Zomato", "BigBasket", "Flipkart",
             "Netflix", "Uber", "HDFC ATM", "Salary Credit", "PhonePe",
             "Myntra", "BookMyShow", "Ola", "Paytm", "Dunzo",
             "IRCTC", "MakeMyTrip", "Nykaa", "Decathlon", "Apple Store"]

txn_statuses = ["Success", "Success", "Success", "Success", "Failed", "Pending"]

# ── INSERT 50 CUSTOMERS ──
customers = []
for i in range(1, 51):
    cust_id = f"CUST{i:03d}"
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    account_type = random.choice(account_types)
    branch = random.choice(branches)

    # Realistic balance ranges by branch
    if branch == "Metro":
        balance = round(random.uniform(20000, 500000), 2)
        credit_score = random.randint(680, 850)
    elif branch == "Urban":
        balance = round(random.uniform(10000, 200000), 2)
        credit_score = random.randint(650, 800)
    elif branch == "Semi Urban":
        balance = round(random.uniform(5000, 100000), 2)
        credit_score = random.randint(620, 750)
    else:  # Rural
        balance = round(random.uniform(1000, 50000), 2)
        credit_score = random.randint(580, 720)

    customers.append((
        cust_id, name, account_type, balance, credit_score, branch,
        f"{name.split()[0].lower()}{i}@email.com",
        f"+91-98765{i:05d}"
    ))

cursor.executemany(
    "INSERT INTO customers VALUES (?,?,?,?,?,?,?,?)", customers)
print(f" Inserted 50 customers")

# ── INSERT 500 TRANSACTIONS ──
transactions = []
for i, (cust_id, *_) in enumerate(customers):
    num_txns = random.randint(8, 12)  # 8-12 transactions per customer
    for j in range(num_txns):
        date = datetime.now() - timedelta(days=random.randint(1, 90))
        txn_type = random.choices(
            ["Debit", "Credit"], weights=[75, 25])[0]
        amount = round(random.uniform(100, 50000), 2)
        transactions.append((
            f"TXN{i+1:03d}{j:03d}",
            cust_id,
            date.strftime("%Y-%m-%d"),
            amount,
            txn_type,
            random.choice(merchants),
            random.choice(txn_statuses)
        ))

cursor.executemany(
    "INSERT INTO transactions VALUES (?,?,?,?,?,?,?)", transactions)
print(f" Inserted {len(transactions)} transactions")

# ── INSERT 50 CREDIT CARDS ──
credit_cards = []
for i, (cust_id, _, _, _, credit_score, branch, *__) in enumerate(customers):
    card_type = random.choice(card_types)

    # Credit limit based on credit score
    if credit_score >= 780:
        credit_limit = round(random.uniform(300000, 800000), 2)
    elif credit_score >= 720:
        credit_limit = round(random.uniform(150000, 300000), 2)
    elif credit_score >= 670:
        credit_limit = round(random.uniform(75000, 150000), 2)
    else:
        credit_limit = round(random.uniform(20000, 75000), 2)

    outstanding = round(random.uniform(0, credit_limit * 0.7), 2)
    minimum_due = round(outstanding * 0.05, 2)
    due_date = (datetime.now() + timedelta(
        days=random.randint(5, 30))).strftime("%Y-%m-%d")
    status = random.choice(card_statuses)

    credit_cards.append((
        f"CARD{i+1:03d}", cust_id, card_type,
        credit_limit, outstanding, minimum_due,
        due_date, status
    ))

cursor.executemany(
    "INSERT INTO credit_cards VALUES (?,?,?,?,?,?,?,?)", credit_cards)
print(f" Inserted 50 credit cards")

conn.commit()
conn.close()

print("\n Database ready!")
print("   → 50 customers (Metro, Urban, Semi Urban, Rural)")
print("   → 500 transactions (mix of Success, Failed, Pending)")
print("   → 50 credit cards (Classic, Platinum, Titanium)")
print("   → Realistic credit scores, balances & limits by branch")

 Tables created!
 Inserted 50 customers
 Inserted 511 transactions
 Inserted 50 credit cards

 Database ready!
   → 50 customers (Metro, Urban, Semi Urban, Rural)
   → 500 transactions (mix of Success, Failed, Pending)
   → 50 credit cards (Classic, Platinum, Titanium)
   → Realistic credit scores, balances & limits by branch


### Tables data preview

In [32]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('banking.db')

# ── CUSTOMERS ──
print("=" * 60)
print(" CUSTOMERS TABLE (first 5 rows)")
print("=" * 60)
df_customers = pd.read_sql("SELECT * FROM customers LIMIT 5", conn)
print(df_customers.to_string(index=False))

# ── TRANSACTIONS ──
print("\n" + "=" * 60)
print(" TRANSACTIONS TABLE (first 5 rows)")
print("=" * 60)
df_txns = pd.read_sql("SELECT * FROM transactions LIMIT 5", conn)
print(df_txns.to_string(index=False))

# ── CREDIT CARDS ──
print("\n" + "=" * 60)
print(" CREDIT CARDS TABLE (first 5 rows)")
print("=" * 60)
df_cards = pd.read_sql("SELECT * FROM credit_cards LIMIT 5", conn)
print(df_cards.to_string(index=False))

# ── SUMMARY STATS ──
print("\n" + "=" * 60)
print(" DATABASE SUMMARY")
print("=" * 60)
print(pd.read_sql("SELECT branch, COUNT(*) as customers, ROUND(AVG(balance),2) as avg_balance, ROUND(AVG(credit_score),0) as avg_credit_score FROM customers GROUP BY branch", conn).to_string(index=False))
print()
print(pd.read_sql("SELECT status, COUNT(*) as count FROM transactions GROUP BY status", conn).to_string(index=False))
print()
print(pd.read_sql("SELECT card_type, COUNT(*) as count, ROUND(AVG(credit_limit),2) as avg_limit FROM credit_cards GROUP BY card_type", conn).to_string(index=False))

conn.close()

 CUSTOMERS TABLE (first 5 rows)
customer_id          name account_type  balance  credit_score     branch               email          phone
    CUST001    Ajay Patel      Savings 31127.79           677 Semi Urban     ajay1@email.com +91-9876500001
    CUST002   Kavya Desai      Savings 75363.44           642 Semi Urban    kavya2@email.com +91-9876500002
    CUST003 Vasantha Shah      Savings 64973.72           739      Metro vasantha3@email.com +91-9876500003
    CUST004  Prakash Kaur      Savings 47779.15           789      Urban  prakash4@email.com +91-9876500004
    CUST005  Manoj Pillai      Current 62856.23           651      Urban    manoj5@email.com +91-9876500005

 TRANSACTIONS TABLE (first 5 rows)
   txn_id customer_id       date   amount   type      merchant  status
TXN001000     CUST001 2026-01-07 42650.72  Debit       Netflix Success
TXN001001     CUST001 2026-01-03  9372.67  Debit   Apple Store Success
TXN001002     CUST001 2025-12-13 20248.40 Credit     Decathlon Success


**Observation-**

For this prototype I used **SQLite** for portability, but the SQL agent is database-agnostic and would connect to PostgreSQL in production with just a connection string change.

In [37]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

# Connect LangChain to our SQLite database
db = SQLDatabase.from_uri("sqlite:///banking.db")

# Verify connection
print(" Database connected!")
print(f" Tables found: {db.get_usable_table_names()}")

 Database connected!
 Tables found: ['credit_cards', 'customers', 'transactions']


### The SQL Agent

In [40]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

# SQL agent with ReAct style — compatible with Groq/Llama
sql_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="zero-shot-react-description",
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

print("SQL Agent ready!")

SQL Agent ready!


**Observation -**

When we use this model, the agent is stuck in a loop — it keeps listing tables but never progresses to actually querying them. This is a known **limitation** of Llama 3.1 8B — it's too small for complex SQL agent reasoning. The fix is simple — upgrade to a larger model just for the SQL agent.

So now, for RAG Agent, we use **Llama 3.1 8B** because Simple task — just summarise retrieved text. Speed matters.

For SQL Agent, we use **Llama 3.3 70B** because Complex task — multi-step reasoning, schema understanding, SQL generation. Accuracy matters.

In [42]:
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

# Larger model for SQL agent — 70B handles ReAct reasoning much better
llm_70b = ChatGroq(
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # Load from .env
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# Rebuild SQL agent with 70B model
sql_agent = create_sql_agent(
    llm=llm_70b,
    db=db,
    agent_type="zero-shot-react-description",
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print("SQL Agent ready with Llama 3.3 70B!")

SQL Agent ready with Llama 3.3 70B!


**Observation-**

I set **max_iterations=10** as a guardrail to prevent runaway agent loops. In production, unbounded agent execution is a real cost and reliability concern — every iteration is an LLM API call. Five iterations is sufficient for our banking query complexity while protecting against failure modes."

### Testing SQL Agent

In [43]:
# Test one query first
result = sql_agent.invoke(
    {"input": "How many customers are there in each branch?"}
)
print(f" {result['output']}")



> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: credit_cards, customers, transactionsWith the list of tables, I can see that the "customers" table is likely to have the information I need. However, I also need to know how the customers are related to the branches. I should query the schema of the "customers" table to see if it has a column related to branches.

Action: sql_db_schema
Action Input: customers
CREATE TABLE customers (
	customer_id TEXT, 
	name TEXT, 
	account_type TEXT, 
	balance REAL, 
	credit_score INTEGER, 
	branch TEXT, 
	email TEXT, 
	phone TEXT, 
	PRIMARY KEY (customer_id)
)

/*
3 rows from customers table:
customer_id	name	account_type	balance	credit_score	branch	email	phone
CUST001	Ajay Patel	Savings	31127.79	677	Semi Urban	ajay1@email.com	+91-9876500001
CUST002	Kavya Desai	Savings	75363.44	

In [44]:
test_queries = [
    "What are the top 5 customers with highest credit card outstanding balance?",
    "How many transactions have Failed status?",
    "What is the average balance of customers by account type?",
    "Which merchant has the highest number of transactions?",
    "Which customers have a blocked or overdue credit card?"
]

print(" Testing SQL Agent across all query types:\n")
for query in test_queries:
    print(f"{'='*60}")
    print(f" {query}")
    print(f"{'='*60}")
    result = sql_agent.invoke({"input": query})
    print(f" {result['output']}\n")

 Testing SQL Agent across all query types:

 What are the top 5 customers with highest credit card outstanding balance?


> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: credit_cards, customers, transactionsWith the list of tables, I can now query the schema of the most relevant tables to find the columns I need to answer the question.

Action: sql_db_schema
Action Input: customers, credit_cards
CREATE TABLE credit_cards (
	card_id TEXT, 
	customer_id TEXT, 
	card_type TEXT, 
	credit_limit REAL, 
	outstanding REAL, 
	minimum_due REAL, 
	due_date TEXT, 
	status TEXT, 
	PRIMARY KEY (card_id), 
	FOREIGN KEY(customer_id) REFERENCES customers (customer_id)
)

/*
3 rows from credit_cards table:
card_id	customer_id	card_type	credit_limit	outstanding	minimum_due	due_date	status
CARD001	CUST001	Classic	109316.28	20075.65	1

**Observations -**

Both agents are now ready:

-  **RAG Agent**  — answers policy questions from 6 HDFC PDFs
-  **SQL Agent**  — answers data questions from 3 database tables

### The LangGraph Orchestrator

In [45]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# ── DEFINE STATE ──
# This is the shared memory that flows through the entire graph
class AgentState(TypedDict):
    query: str          # Original user question
    agent_used: str     # Which agent handled it
    response: str       # Final answer

# ── NODE 1: ROUTER ──
# Decides which agent should handle the query
def router(state: AgentState) -> AgentState:
    query = state["query"].lower()

    # SQL keywords → personal/transactional data questions
    sql_keywords = [
        "transaction", "balance", "how many", "outstanding",
        "credit card", "merchant", "customer", "branch",
        "average", "total", "count", "highest", "lowest",
        "failed", "blocked", "overdue", "statement"
    ]

    # RAG keywords → policy/rule questions
    rag_keywords = [
        "policy", "rule", "guideline", "what is", "how does",
        "eligibility", "penalty", "interest rate", "fee",
        "grievance", "kyc", "document", "complaint", "process",
        "minimum balance", "loan", "terms", "conditions"
    ]

    sql_score = sum(1 for kw in sql_keywords if kw in query)
    rag_score = sum(1 for kw in rag_keywords if kw in query)

    if sql_score > rag_score:
        state["agent_used"] = "sql"
    else:
        state["agent_used"] = "rag"

    print(f" Router Decision: {state['agent_used'].upper()} Agent "
          f"(SQL score: {sql_score}, RAG score: {rag_score})")
    return state

# ── NODE 2: RAG AGENT NODE ──
def run_rag_agent(state: AgentState) -> AgentState:
    print(f" RAG Agent processing...")
    response = rag_chain.invoke(state["query"])
    state["response"] = response
    return state

# ── NODE 3: SQL AGENT NODE ──
def run_sql_agent(state: AgentState) -> AgentState:
    print(f" SQL Agent processing...")
    result = sql_agent.invoke({"input": state["query"]})
    state["response"] = result["output"]
    return state

# ── CONDITIONAL EDGE: Which agent to call? ──
def route_to_agent(state: AgentState) -> str:
    return state["agent_used"]  # Returns "sql" or "rag"

print(" Agent nodes defined!")

 Agent nodes defined!


In [46]:
# ── BUILD THE GRAPH ──
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("router", router)
workflow.add_node("rag", run_rag_agent)
workflow.add_node("sql", run_sql_agent)

# Set entry point
workflow.set_entry_point("router")

# Add conditional edges from router
workflow.add_conditional_edges(
    "router",
    route_to_agent,
    {
        "rag": "rag",
        "sql": "sql"
    }
)

# Both agents lead to END
workflow.add_edge("rag", END)
workflow.add_edge("sql", END)

# Compile the graph
app = workflow.compile()

print(" LangGraph Orchestrator compiled!")
print("""
Graph Structure:
    START
      ↓
   [ROUTER] ── decides ──→ [RAG AGENT]  → END
                      └──→ [SQL AGENT]  → END
""")

 LangGraph Orchestrator compiled!

Graph Structure:
    START
      ↓
   [ROUTER] ── decides ──→ [RAG AGENT]  → END
                      └──→ [SQL AGENT]  → END



### Full end-to-end test — queries should route to different agents

In [47]:
# Full end-to-end test — queries should route to different agents
test_queries = [
    "What is the penalty for missed loan EMI payment?",      # → RAG
    "Which customers have overdue credit cards?",            # → SQL
    "What documents are needed for KYC verification?",       # → RAG
    "Which merchant has the highest number of transactions?", # → SQL
    "How can I raise a grievance against HDFC Bank?",        # → RAG
]

def ask(query):
    print(f"\n{'='*90}")
    print(f" {query}")
    print(f"{'='*90}")
    result = app.invoke({
        "query": query,
        "agent_used": "",
        "response": ""
    })
    print(f" {result['response']}")

for query in test_queries:
    ask(query)


 What is the penalty for missed loan EMI payment?
 Router Decision: RAG Agent (SQL score: 0, RAG score: 3)
 RAG Agent processing...
 I don't have enough information in the policy documents to answer this. Please contact HDFC Bank directly.

 Which customers have overdue credit cards?
 Router Decision: SQL Agent (SQL score: 3, RAG score: 0)
 SQL Agent processing...


> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: credit_cards, customers, transactionsWith the list of tables, I can now query the schema of the most relevant tables to find out the column names and data types.

Action: sql_db_schema
Action Input: credit_cards, customers
CREATE TABLE credit_cards (
	card_id TEXT, 
	customer_id TEXT, 
	card_type TEXT, 
	credit_limit REAL, 
	outstanding REAL, 
	minimum_due REAL, 
	due_date TEXT, 
	status TEXT, 
	PRIMARY 